In [9]:
import torch

tensors_dir = "../data/tensors"

X_trainval = torch.load(f"{tensors_dir}/X_trainval.pt").float()
y_trainval = torch.load(f"{tensors_dir}/y_trainval.pt").long()
sample_weights = torch.load(f"{tensors_dir}/sample_weights.pt").float()

print(f"X_train: {X_trainval.shape}  {X_trainval.dtype}")
print(f"y_train: {y_trainval.shape}  {y_trainval.dtype}")
print(f"weights: {sample_weights.shape}")

X_train: torch.Size([796, 10])  torch.float32
y_train: torch.Size([796])  torch.int64
weights: torch.Size([796])


In [6]:
import optuna

study = optuna.create_study(
    study_name="test",
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2),
    storage="sqlite:///../runs/optuna/study.db",
    load_if_exists=True,
)

[I 2026-06-17 11:33:54,786] A new study created in RDB with name: test


In [7]:
from src import make_objective

INPUT_SIZE = X_trainval.shape[1]
NUM_CLASSES = 4

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

objective = make_objective(X=X_trainval, y=y_trainval, weights=sample_weights, n_epochs=100, n_splits=5,
               input_size=INPUT_SIZE, output_size=NUM_CLASSES, device=device)

study.optimize(objective, n_trials=50, show_progress_bar=True)

  0%|          | 0/50 [00:00<?, ?it/s]

[Fold 0] Epoch   0/100 | train: 1.4297 | val: 1.4036 <- new best
[Fold 0] Epoch   1/100 | train: 1.4236 | val: 1.4036 <- new best
[Fold 0] Epoch   2/100 | train: 1.4330 | val: 1.4036 <- new best
[Fold 0] Epoch   3/100 | train: 1.4462 | val: 1.4036 <- new best
[Fold 0] Epoch   4/100 | train: 1.4567 | val: 1.4036 <- new best
[Fold 0] Epoch   5/100 | train: 1.4314 | val: 1.4036
[Fold 0] Epoch   6/100 | train: 1.4443 | val: 1.4036
[Fold 0] Epoch   7/100 | train: 1.4347 | val: 1.4036
[Fold 0] Epoch   8/100 | train: 1.4300 | val: 1.4036
[Fold 0] Epoch   9/100 | train: 1.4473 | val: 1.4036
[Fold 0] Epoch  10/100 | train: 1.4356 | val: 1.4036
[Fold 0] Epoch  11/100 | train: 1.4216 | val: 1.4036
[Fold 0] Epoch  12/100 | train: 1.4272 | val: 1.4036
[Fold 0] Epoch  13/100 | train: 1.4406 | val: 1.4036
[Fold 0] Epoch  14/100 | train: 1.4429 | val: 1.4036
[Fold 0] Epoch  15/100 | train: 1.4105 | val: 1.4036
[Fold 0] Epoch  16/100 | train: 1.4296 | val: 1.4036
[Fold 0] Epoch  17/100 | train: 1.4173 

In [8]:
print(f"Best trial:  #{study.best_trial.number}")
print(f"\nBest val loss: {study.best_value:.4f}")
print(f"Hiperparameters:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

Best trial:  #36

Best val loss: 1.1797
Hiperparameters:
  batch_size: 16
  n_hidden: 1
  hidden_dim_l0: 256
  dropout_p: 0.3582812726459808
  activation: relu
  optimizer: Adam
  lr: 0.0013305774914038324
